# Smart MCQ Solver Challenge
**Roll No:** 23f3004491 | **Term:** T2-2026 | **Metric:** MAP@3 | **Cutoff:** 0.73

Each question has a prompt + 5 options (A-E); predict the top-3 answers in ranked order.

### Key data insights (drive all model choices)
1. Distractor options are **minimal edits** of each other (~34% of questions have options
   >70% character-identical) - similarity ranking fails; options must be compared side-by-side.
2. Questions test **factual science knowledge** - small encoders lack it; a 7B LLM contains it.
3. **Train contains only 962 unique questions in 2000 rows** (same question re-wrapped in
   different phrasings), and **~81% of test questions duplicate a train question** - enabling
   a lookup strategy, and requiring a dedup-aware validation split to avoid leakage.

### Model progression (validation MAP@3)
| # | Model | Type | MAP@3 |
|---|---|---|---|
| 1 | TF-IDF + cosine | from scratch | 0.31 |
| 2 | MiniLM bi-encoder | pretrained | 0.40 |
| 3 | NLI cross-encoder | pretrained, zero-shot | 0.56 |
| 4 | LoRA DeBERTa-v3-large (multiple-choice) | fine-tuned | 0.59 |
| 5 | Qwen2.5-7B-Instruct | large LLM, zero-shot | 0.90* |
| 6 | **Hybrid: train-lookup + Qwen** | final submission | **leaderboard** |

*inflated by duplicate leakage in the naive split; see Section 5 for the honest split.

## 1. Environment Setup

In [1]:
!pip install -q -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" "sentence-transformers==3.3.1"
print("Packages installed. RESTART kernel, then Run All.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 615.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 52.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 whic

In [2]:
import warnings
warnings.filterwarnings('ignore')

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

OPTIONS = ['A', 'B', 'C', 'D', 'E']
SEED = 42

GPU available: True
GPU name: Tesla T4


## 2. Load Data

In [3]:
import pandas as pd

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train  = pd.read_csv(f"{BASE}/train.csv")
test   = pd.read_csv(f"{BASE}/test.csv")
sample = pd.read_csv(f"{BASE}/sample_submission.csv")

print("TRAIN:", train.shape, "| TEST:", test.shape)

TRAIN: (2000, 8) | TEST: (500, 7)


## 3. Quick EDA — the two facts that matter

1. **Option near-duplication** — distractors are minimal edits, so the discriminating
   signal is one or two tokens. This kills similarity approaches.
2. **Answer distribution** — mildly skewed to B/C; sets the naive floor (~0.30 MAP@3).

In [4]:
import numpy as np
import difflib
from itertools import combinations

sims = []
for i in range(300):
    r = train.iloc[i]
    pair_sims = [difflib.SequenceMatcher(None, str(r[a]), str(r[b])).ratio()
                 for a, b in combinations(OPTIONS, 2)]
    sims.append(np.mean(pair_sims))
sims = np.array(sims)
print(f"Mean pairwise option similarity: {sims.mean():.3f}")
print(f"Questions with near-identical options (>0.7): {(sims > 0.7).mean()*100:.0f}%")

print("\nAnswer distribution:")
print(train['answer'].value_counts().sort_index().to_string())

total_len = train['prompt'].str.split().str.len() + train[OPTIONS].apply(
    lambda col: col.str.split().str.len()).max(axis=1)
print(f"\nPrompt+longest-option words: median={total_len.median():.0f}, "
      f"95th pct={total_len.quantile(0.95):.0f}, max={total_len.max():.0f}")

Mean pairwise option similarity: 0.588
Questions with near-identical options (>0.7): 34%

Answer distribution:
answer
A    369
B    490
C    459
D    358
E    324

Prompt+longest-option words: median=45, 95th pct=84, max=148


## 4. MAP@3 Evaluation Metric

Correct answer at rank 1 → 1.0, rank 2 → 0.5, rank 3 → 0.333, else 0.

In [5]:
def average_precision_at_3(true_label, predicted_labels):
    """Score one question: 1/(rank+1) if the correct answer is in the top-3, else 0."""
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    """Average AP@3 across all questions."""
    return np.mean([average_precision_at_3(t, p)
                    for t, p in zip(true_labels, predicted_lists)])

assert average_precision_at_3('B', ['B','A','C']) == 1.0
assert average_precision_at_3('B', ['A','B','C']) == 0.5
assert abs(average_precision_at_3('B', ['A','C','B']) - 1/3) < 1e-9
assert average_precision_at_3('B', ['A','C','D']) == 0.0
print("MAP@3 scorer verified.")

MAP@3 scorer verified.


## 5. Prompt Normalization & Leakage-Free Split

The same core question appears multiple times in train with different wrapper phrases
("Pick the best possible answer:" vs "Select the most accurate option:"). A naive random
split puts copies of the same question on both sides, inflating validation (we measured
0.90 vs a 0.69 leaderboard). Fix: strip wrappers to get the **core question**, then split
by unique core so no question appears in both train and validation.

In [6]:
import re
from sklearn.model_selection import train_test_split

WRAPPERS_START = ["pick the best possible answer:", "select the most accurate option:",
    "identify the correct statement:", "determine the correct option:", "choose the right answer:",
    "answer the following:", "choose the correct option:", "select the correct answer:",
    "pick the correct option:", "identify the right option:"]
WRAPPERS_END = ["among the listed options.", "among the listed options", "carefully.", "carefully",
    "from the choices below.", "from the choices below", "from the options given.", "from the options given"]

def normalize_prompt(p):
    """Strip wrapper phrases to recover the core question text."""
    p = str(p).strip().lower()
    changed = True
    while changed:
        changed = False
        for w in WRAPPERS_START:
            if p.startswith(w):
                p = p[len(w):].strip(); changed = True
        for w in WRAPPERS_END:
            if p.endswith(w):
                p = p[:-len(w)].strip(); changed = True
    return re.sub(r'\s+', ' ', p)

train['core'] = train['prompt'].apply(normalize_prompt)
test['core'] = test['prompt'].apply(normalize_prompt)

unique_cores = train['core'].unique()
core_train, core_valid = train_test_split(unique_cores, test_size=0.2, random_state=SEED)

train_df = train[train['core'].isin(core_train)].reset_index(drop=True)
valid_df = train[train['core'].isin(core_valid)].drop_duplicates('core').reset_index(drop=True)

print("Unique core questions:", len(unique_cores), "of", len(train), "rows")
print("Train rows:", len(train_df), "| Valid rows (deduped):", len(valid_df))
print("Core overlap between splits:", len(set(train_df['core']) & set(valid_df['core'])))

Unique core questions: 962 of 2000 rows
Train rows: 1596 | Valid rows (deduped): 193
Core overlap between splits: 0


## 6. W&B Setup

Every model below logs a run so all runs are comparable on the same metrics (MAP@3, accuracy, F1). API key lives in a Kaggle Secret — safe to commit this notebook.

In [7]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "23f3004491-t22026"
wandb.login(key=os.environ["WANDB_API_KEY"])

from sklearn.metrics import accuracy_score, f1_score

def log_model_run(name, model_type, valid_df, predictions, config=None):
    """Log one model's validation performance to W&B as its own run.

    Logs MAP@3 plus top-1 accuracy and macro-F1 so all runs share common
    metrics for comparison (a project requirement).
    """
    truth = valid_df['answer'].tolist()
    top1  = [p[0] for p in predictions]

    metrics = {
        "map@3":      mean_average_precision_at_3(truth, predictions),
        "accuracy":   accuracy_score(truth, top1),
        "f1_macro":   f1_score(truth, top1, average='macro'),
    }

    run = wandb.init(project=os.environ["WANDB_PROJECT"], name=name,
                     config={"model_type": model_type, **(config or {})},
                     reinit=True)
    wandb.log(metrics)
    run.finish()

    print(f"[{name}] " + " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    return metrics["map@3"]

print("W&B ready.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3004491 (23f3004491-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready.


## 7. Model 1 — TF-IDF + Cosine Similarity *(from scratch)*

Word-frequency vectors, no semantics. Expected weak: when options differ by one word,
their TF-IDF vectors are nearly identical.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_predict(df):
    """Rank options by TF-IDF cosine similarity to the prompt."""
    predictions = []
    for _, row in df.iterrows():
        texts = [row['prompt']] + [row[o] for o in OPTIONS]
        vectors = TfidfVectorizer(stop_words='english').fit_transform(texts)
        scores = cosine_similarity(vectors[0], vectors[1:])[0]
        predictions.append([OPTIONS[i] for i in np.argsort(scores)[::-1]])
    return predictions

tfidf_preds = tfidf_predict(valid_df)
tfidf_score = log_model_run("tfidf-cosine", "from-scratch", valid_df, tfidf_preds)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260716_182435-pyy0nval
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-cosine
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/pyy0nval
wandb: updating run metadata; uploading summary
wandb: updating run metadata
wandb: uploading wandb-metadata.json; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.16062
wandb: f1_macro 0.15818
wandb:    map@3 0.3247
wandb: 
wandb: 🚀 View run tfidf-cosine at: https:

[tfidf-cosine] map@3=0.3247 | accuracy=0.1606 | f1_macro=0.1582


## 8. Model 2 — MiniLM Bi-Encoder *(pretrained)*

Semantic embeddings, prompt and options embedded **separately**. Also expected to underperform
here: near-identical option texts → near-identical embeddings → no ranking signal.

In [9]:
from sentence_transformers import SentenceTransformer, util

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_predict(df, model):
    """Rank options by semantic cosine similarity to the prompt."""
    prompt_embs = model.encode(df['prompt'].tolist(), convert_to_tensor=True)
    option_embs = {o: model.encode(df[o].tolist(), convert_to_tensor=True) for o in OPTIONS}

    predictions = []
    for i in range(len(df)):
        scores = [util.cos_sim(prompt_embs[i], option_embs[o][i]).item() for o in OPTIONS]
        predictions.append([OPTIONS[j] for j in np.argsort(scores)[::-1]])
    return predictions

bi_preds = embedding_predict(valid_df, bi_encoder)
bi_score = log_model_run("minilm-bi-encoder", "pretrained-zero-shot", valid_df, bi_preds,
                         config={"base_model": "all-MiniLM-L6-v2"})

2026-07-16 18:24:47.552210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784226287.711706      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784226287.760665      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784226288.137831      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784226288.137871      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784226288.137874      23 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

wandb: setting up run z3jyarl2
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260716_182510-z3jyarl2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run minilm-bi-encoder
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/z3jyarl2
wandb: updating run metadata; uploading summary
wandb: uploading requirements.txt; uploading wandb-summary.json; uploading wandb-metadata.json
wandb: uploading requirements.txt; uploading wandb-summary.json
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.27461
wandb: f1_macro 0.27114
wandb:    map@3 0.41969
wandb: 
wandb: 🚀 View run minilm-bi-encoder at: https://wandb.ai/23f3004491-indian-instit

[minilm-bi-encoder] map@3=0.4197 | accuracy=0.2746 | f1_macro=0.2711


## 9. Model 3 — NLI Cross-Encoder *(pretrained, zero-shot)*

Reads (prompt, option) **together** via cross-attention and scores entailment.
First model that can attend to the one-word differences between options.

In [10]:
from sentence_transformers import CrossEncoder

ce_model = CrossEncoder('cross-encoder/nli-deberta-v3-small')

def cross_encoder_predict(df, model):
    """Rank options by NLI entailment score against the prompt."""
    all_pairs = [(row['prompt'], row[o]) for _, row in df.iterrows() for o in OPTIONS]
    logits = model.predict(all_pairs, batch_size=64)
    entail = logits[:, 1].reshape(len(df), 5)

    return [[OPTIONS[j] for j in np.argsort(scores)[::-1]] for scores in entail]

ce_preds = cross_encoder_predict(valid_df, ce_model)
ce_score = log_model_run("nli-cross-encoder", "pretrained-zero-shot", valid_df, ce_preds,
                         config={"base_model": "cross-encoder/nli-deberta-v3-small"})

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

wandb: setting up run 770imdbk
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260716_182523-770imdbk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run nli-cross-encoder
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/770imdbk
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-metadata.json; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.41969
wandb: f1_macro 0.40949
wandb:    map@3 0.58377
wandb: 
wandb: 🚀 View

[nli-cross-encoder] map@3=0.5838 | accuracy=0.4197 | f1_macro=0.4095


## 10. Model 4 — Multiple-Choice LoRA Fine-Tune *(model of choice)*

### Why this architecture
Previous fine-tuning scored each (prompt, option) pair **independently** — but with
minimal-edit distractors, correctness is only defined *relative to the alternatives*.
`AutoModelForMultipleChoice` encodes all 5 (prompt, option) pairs and applies a
**softmax across the 5 options**, forcing a direct comparison. Side benefits:
- No 80/20 class imbalance (the label is just the correct option index 0–4).
- Directly optimizes ranking — which is what MAP@3 measures.

### Why DeBERTa-v3-large
The questions test factual science knowledge; the 140M-param small model plateaued
because it lacks that knowledge. The 400M large variant is the standard backbone for
this task family. Fitting it on a T4 requires small batches + gradient accumulation
(= Milestone 4's "managing GPU memory and batch sizes").

### 10.1 Tokenize in multiple-choice format
Each example becomes 5 parallel (prompt, option) encodings, label = index of the answer.

In [11]:
from transformers import AutoTokenizer
from datasets import Dataset

MC_MODEL = "microsoft/deberta-v3-large"
MAX_LEN  = 192

tokenizer = AutoTokenizer.from_pretrained(MC_MODEL)

def preprocess_mc(examples):
    """Tokenize one batch into multiple-choice format:
    input_ids shape (batch, 5, seq_len), label = correct option index."""
    first  = [[p] * 5 for p in examples['prompt']]
    second = [[examples[o][i] for o in OPTIONS]
              for i in range(len(examples['prompt']))]

    flat_first  = sum(first, [])
    flat_second = sum(second, [])
    tok = tokenizer(flat_first, flat_second,
                    truncation=True, max_length=MAX_LEN, padding='max_length')

    grouped = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tok.items()}
    if 'answer' in examples:
        grouped['labels'] = [OPTIONS.index(a) for a in examples['answer']]
    return grouped

def to_mc_dataset(df, with_labels=True):
    cols = ['prompt'] + OPTIONS + (['answer'] if with_labels else [])
    ds = Dataset.from_pandas(df[cols])
    ds = ds.map(preprocess_mc, batched=True, remove_columns=ds.column_names)
    ds.set_format('torch')
    return ds

train_mc = to_mc_dataset(train_df)
valid_mc = to_mc_dataset(valid_df)

print("MC datasets ready:", train_mc.num_rows, "train /", valid_mc.num_rows, "valid")
print("input_ids shape per example:", tuple(train_mc[0]['input_ids'].shape), "= (5 options, seq_len)")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1596 [00:00<?, ? examples/s]

Map:   0%|          | 0/193 [00:00<?, ? examples/s]

MC datasets ready: 1596 train / 193 valid
input_ids shape per example: (5, 192) = (5 options, seq_len)


### 10.2 Load DeBERTa-v3-large + LoRA adapters

In [12]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForMultipleChoice.from_pretrained(MC_MODEL)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj",
                    "output.dense", "intermediate.dense"],
    modules_to_save=["classifier", "pooler"],
)

model_mc = get_peft_model(base_model, lora_config)
model_mc.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

trainable params: 8,128,513 || all params: 443,191,298 || trainable%: 1.8341


### 10.3 Training arguments — GPU memory management

DeBERTa-v3-large at seq_len 192 × 5 options is heavy for a 16 GB T4, so:
- `per_device_train_batch_size=2` with `gradient_accumulation_steps=8` → effective batch 16
- `gradient_checkpointing=True` trades compute for memory
- `bf16=True` halves activation memory (and avoids the fp16/LoRA grad-scaler bug)

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mc_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="wandb",
    run_name="mc-lora-deberta-v3-large",
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    save_total_limit=1,
)

print("Training args ready. Effective batch:",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

Training args ready. Effective batch: 16


### 10.4 Train

`compute_metrics` here reports **MAP@3 directly** (plus top-1 accuracy) each epoch,
so the model is selected by the actual competition metric.

In [14]:
from transformers import Trainer

def compute_mc_metrics(eval_pred):
    """Top-1 accuracy + MAP@3 computed from the 5-way logits."""
    logits, labels = eval_pred
    order = np.argsort(-logits, axis=1)
    top1 = order[:, 0]

    ap3 = []
    for row_order, true_idx in zip(order, labels):
        rank = np.where(row_order == true_idx)[0][0]
        ap3.append(1.0 / (rank + 1) if rank < 3 else 0.0)

    return {"accuracy": (top1 == labels).mean(), "map3": np.mean(ap3)}

trainer = Trainer(
    model=model_mc,
    args=training_args,
    train_dataset=train_mc,
    eval_dataset=valid_mc,
    processing_class=tokenizer,
    compute_metrics=compute_mc_metrics,
)

trainer.train()

wandb: setting up run khbhrueg
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260716_182538-khbhrueg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mc-lora-deberta-v3-large
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/khbhrueg


model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

{'loss': 1.6174, 'grad_norm': 21.92616081237793, 'learning_rate': 9.242424242424242e-05, 'epoch': 0.5012531328320802}
{'eval_loss': 1.588219165802002, 'eval_accuracy': 0.40932642487046633, 'eval_map3': 0.586355785837651, 'eval_runtime': 29.2053, 'eval_samples_per_second': 6.608, 'eval_steps_per_second': 0.856, 'epoch': 0.9824561403508771}
{'loss': 1.606, 'grad_norm': 10.7350435256958, 'learning_rate': 7.348484848484849e-05, 'epoch': 1.0100250626566416}
{'loss': 1.6034, 'grad_norm': 12.06516170501709, 'learning_rate': 5.4545454545454546e-05, 'epoch': 1.5112781954887218}
{'eval_loss': 1.580622673034668, 'eval_accuracy': 0.45077720207253885, 'eval_map3': 0.6131260794473229, 'eval_runtime': 29.0705, 'eval_samples_per_second': 6.639, 'eval_steps_per_second': 0.86, 'epoch': 1.9924812030075187}
{'loss': 1.588, 'grad_norm': 10.240534782409668, 'learning_rate': 3.560606060606061e-05, 'epoch': 2.020050125313283}
{'loss': 1.5966, 'grad_norm': 11.483855247497559, 'learning_rate': 1.666666666666666

TrainOutput(global_step=147, training_loss=1.6002130573298656, metrics={'train_runtime': 1243.2585, 'train_samples_per_second': 3.851, 'train_steps_per_second': 0.118, 'train_loss': 1.6002130573298656, 'epoch': 2.962406015037594})

### 10.5 Score with the notebook's MAP@3 (consistency check + W&B run)

In [15]:
from transformers.integrations import WandbCallback
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks if not isinstance(cb, WandbCallback)
]

def mc_predict(df, trainer, with_labels=True):
    """Rank the 5 options per question using the fine-tuned MC model."""
    ds = to_mc_dataset(df, with_labels=with_labels)
    logits = trainer.predict(ds).predictions
    order = np.argsort(-logits, axis=1)
    return [[OPTIONS[j] for j in row] for row in order]

mc_preds = mc_predict(valid_df, trainer)
mc_score = log_model_run("mc-lora-deberta-v3-large", "fine-tuned", valid_df, mc_preds,
                         config={"base_model": MC_MODEL, "r": 16, "lr": 1e-4,
                                 "epochs": 3, "max_len": MAX_LEN})

Map:   0%|          | 0/193 [00:00<?, ? examples/s]

wandb: Finishing previous runs because reinit is set to True.
wandb: updating run metadata
wandb: uploading output.log; uploading config.yaml
wandb: uploading output.log
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁█▅
wandb:               eval/loss █▂▁
wandb:               eval/map3 ▁█▆
wandb:            eval/runtime ▇▁█
wandb: eval/samples_per_second ▂█▁
wandb:   eval/steps_per_second ▂█▁
wandb:             train/epoch ▁▂▂▄▅▅▇██
wandb:       train/global_step ▁▂▂▄▅▅▇██
wandb:         train/grad_norm █▁▂▁▂
wandb:     train/learning_rate █▆▄▃▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.43523
wandb:               eval/loss 1.57866
wandb:               eval/map3 0.60449
wandb:            eval/runtime 29.2396
wandb: eval/samples_per_second 6.601
wandb:   eval/steps_per_second 0.855
wandb:              total_flos 8439904727654400.0
wandb:             train/epoch 2.96241
wandb:       train/global_step 147
wandb:         trai

[mc-lora-deberta-v3-large] map@3=0.6131 | accuracy=0.4508 | f1_macro=0.4404


## 10.6 Model 5 — Qwen2.5-7B-Instruct *(large LLM, zero-shot)* ⭐

**Why this is the model that can cross 0.73.** The encoders plateaued at ~0.58 because the
bottleneck is *factual scientific knowledge* the small models never learned — and LoRA on a
few thousand examples can't inject knowledge. A 7B instruction-tuned LLM already **contains**
this knowledge from pretraining.

**How we score options:** we show the LLM the question and all 5 options, then read the
**probability it assigns to each letter token (A–E)** as the next token. Ranking those five
probabilities gives our top-3 — no generation parsing needed, and it directly yields a ranking
for MAP@3.

**Memory:** the DeBERTa trainer is deleted first to free GPU memory, then Qwen loads in
**fp16 (~14 GB)** — fits a 16 GB T4/P100. Inference only (no training), so fast and stable.
If loading still OOMs, switch `LLM_NAME` to `Qwen/Qwen2.5-3B-Instruct`.

In [16]:
!pip uninstall -q -y bitsandbytes
print("bitsandbytes removed (not needed - fp16 loading instead).")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


bitsandbytes removed (not needed - fp16 loading instead).


In [17]:
import gc
from transformers import AutoModelForCausalLM

del trainer.model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
llm.eval()
print("Qwen2.5-7B loaded in fp16.")

GPU free: 13.1 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B loaded in fp16.


In [18]:
LETTER_IDS = [llm_tokenizer(f" {L}", add_special_tokens=False)['input_ids'][-1]
              for L in OPTIONS]

def build_prompt(row):
    """Format one MCQ into a chat prompt asking for the best letter."""
    opts = "\n".join(f"{L}. {row[L]}" for L in OPTIONS)
    user = (f"Answer the multiple-choice question. Reply with only the letter "
            f"of the single best option.\n\nQuestion: {row['prompt']}\n\n{opts}\n\nAnswer:")
    messages = [{"role": "user", "content": user}]
    return llm_tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True)

@torch.no_grad()
def llm_predict(df, batch_log_every=100):
    """Rank options by the LLM's next-token probability over letters A-E."""
    predictions = []
    for n, (_, row) in enumerate(df.iterrows()):
        text = build_prompt(row)
        inputs = llm_tokenizer(text, return_tensors="pt").to(llm.device)
        logits = llm(**inputs).logits[0, -1]
        letter_logits = logits[LETTER_IDS].float().cpu().numpy()
        order = np.argsort(letter_logits)[::-1]
        predictions.append([OPTIONS[i] for i in order])
        if (n + 1) % batch_log_every == 0:
            print(f"  {n+1}/{len(df)} scored")
    return predictions

llm_preds = llm_predict(valid_df)
llm_score = log_model_run("qwen2.5-7b-zeroshot", "large-llm-zero-shot", valid_df, llm_preds,
                          config={"base_model": LLM_NAME, "quant": "4bit-nf4"})

  100/193 scored


wandb: setting up run kp02l4z9
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260716_184953-kp02l4z9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qwen2.5-7b-zeroshot
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/kp02l4z9
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json
wandb: uploading wandb-summary.json
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.84974
wandb: f1_macro 0.83019
wandb:    map@3 0.91883
wandb: 
wandb: 🚀 View run qwen2.5-7b-zeroshot at: https://wandb.ai/23f3004491-indian-insti

[qwen2.5-7b-zeroshot] map@3=0.9188 | accuracy=0.8497 | f1_macro=0.8302


## 11. Model Comparison

In [19]:
results = pd.DataFrame([
    {"Model": "TF-IDF + cosine",            "Type": "from scratch", "MAP@3": tfidf_score},
    {"Model": "MiniLM bi-encoder",          "Type": "pretrained",   "MAP@3": bi_score},
    {"Model": "NLI cross-encoder",          "Type": "zero-shot",    "MAP@3": ce_score},
    {"Model": "MC LoRA DeBERTa-v3-large",   "Type": "fine-tuned",   "MAP@3": mc_score},
    {"Model": "Qwen2.5-7B-Instruct",        "Type": "large LLM",    "MAP@3": llm_score},
]).sort_values("MAP@3", ascending=False)

print(results.to_string(index=False))
print("\nCutoff to beat: 0.73")

                   Model         Type    MAP@3
     Qwen2.5-7B-Instruct    large LLM 0.918826
MC LoRA DeBERTa-v3-large   fine-tuned 0.613126
       NLI cross-encoder    zero-shot 0.583765
       MiniLM bi-encoder   pretrained 0.419689
         TF-IDF + cosine from scratch 0.324698

Cutoff to beat: 0.73


## 12. Retrieval-Based Duplicate Answering + Final Hybrid Submission

**Data finding (documented in Section 5):** the train file contains each core question in
multiple wrapper phrasings, and ~80% of test questions are re-wrapped duplicates of train
questions. Answering those via retrieval from the labeled training data is a legitimate
retrieval-augmentation strategy - documented transparently here and in the report.

**Precision safeguards** (first submission with naive fuzzy matching scored 0.734,
suggesting fuzzy matches on minimal-edit distractors are error-prone):
- **Exact answer-text match** -> high precision -> lookup answer ranked first.
- **Fuzzy match** (>0.9 similarity AND dominant by >0.05 over second-best) -> ranked
  second behind Qwen's top pick, cushioning potential errors.
- **Ambiguous or novel** -> Qwen's full ranking.

In [20]:
import difflib

train['answer_text'] = train.apply(lambda r: str(r[r['answer']]), axis=1)
answer_lookup = dict(zip(train['core'], train['answer_text']))

def lookup_with_confidence(row):
    """Return (option_letter, 'exact'|'fuzzy') for a retrieved duplicate, else (None, None)."""
    if row['core'] not in answer_lookup:
        return None, None
    ans = answer_lookup[row['core']].strip()
    for o in OPTIONS:
        if str(row[o]).strip() == ans:
            return o, 'exact'
    sims = sorted(((difflib.SequenceMatcher(None, str(row[o]).strip().lower(), ans.lower()).ratio(), o)
                   for o in OPTIONS), reverse=True)
    (s1, o1), (s2, _) = sims[0], sims[1]
    if s1 > 0.9 and (s1 - s2) > 0.05:
        return o1, 'fuzzy'
    return None, None

matches = test.apply(lookup_with_confidence, axis=1)
test['lookup_answer'] = [m[0] for m in matches]
test['lookup_kind'] = [m[1] for m in matches]
print(test['lookup_kind'].value_counts(dropna=False).to_string())

lookup_kind
exact    362
None     111
fuzzy     27


In [21]:
wandb.finish()

llm_test_preds = llm_predict(test)

final_preds = []
for i, (_, row) in enumerate(test.iterrows()):
    qwen_rank = llm_test_preds[i]
    la, kind = row['lookup_answer'], row['lookup_kind']
    if kind == 'exact':
        rest = [o for o in qwen_rank if o != la]
        final_preds.append([la] + rest[:2])
    elif kind == 'fuzzy':
        top = qwen_rank[0]
        if la == top:
            final_preds.append(qwen_rank[:3])
        else:
            rest = [o for o in qwen_rank if o not in (top, la)]
            final_preds.append([top, la] + rest[:1])
    else:
        final_preds.append(qwen_rank[:3])

submission = pd.DataFrame({
    "ID": test['id'],
    "Prediction": [" ".join(p[:3]) for p in final_preds]
})
submission.to_csv("submission.csv", index=False)

assert list(submission.columns) == ["ID", "Prediction"]
assert submission['Prediction'].str.split().str.len().eq(3).all()
assert len(submission) == len(test)
print(submission.head())
print("\nRows:", len(submission), "- confidence-aware hybrid submission ready.")

  100/500 scored
  200/500 scored
  300/500 scored
  400/500 scored
  500/500 scored
   ID Prediction
0   1      A E D
1   2      B A E
2   3      B C E
3   4      E B C
4   5      C A D

Rows: 500 - confidence-aware hybrid submission ready.


## 13. (Optional) Ensemble — Milestone 5

If two models are individually strong, averaging their per-option scores can beat either alone.
Here we average normalized ranks from the LLM and the cross-encoder. Only worth submitting if
it beats the best single model on validation.

In [22]:
def rank_scores(predictions):
    """Convert each ranked letter-list into a per-option score (higher = better)."""
    score_rows = []
    for pred in predictions:
        s = {L: 0 for L in OPTIONS}
        for rank, L in enumerate(pred):
            s[L] = len(OPTIONS) - rank
        score_rows.append([s[L] for L in OPTIONS])
    return np.array(score_rows, dtype=float)

def ensemble(pred_lists, weights):
    """Weighted-average several models' rank-scores, then re-rank."""
    combined = sum(w * rank_scores(p) for w, p in zip(weights, pred_lists))
    return [[OPTIONS[i] for i in np.argsort(row)[::-1]] for row in combined]

ens_val = ensemble([llm_preds, ce_preds], weights=[0.7, 0.3])
ens_score = mean_average_precision_at_3(valid_df['answer'].tolist(), ens_val)
print(f"Ensemble (LLM 0.7 + CE 0.3) MAP@3: {ens_score:.4f}")
print(f"Best single (LLM):               {llm_score:.4f}")
print("Use ensemble?" , "YES" if ens_score > llm_score else "no - LLM alone is better")

Ensemble (LLM 0.7 + CE 0.3) MAP@3: 0.8877
Best single (LLM):               0.9188
Use ensemble? no - LLM alone is better
